# IOAI — 2025 Contest Cluster Pictures (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/data_1.npz'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-cluster-pictures', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/data_1.npz' if os.path.exists('data/data_1.npz') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Cluster Pictures · 모범답안

**핵심 통찰**: 인자분해 `data_1·data_2` 는 비유일하지만 **곱(재구성 이미지)은 유일**합니다. 그러므로 원시 인자가
아니라 **재구성한 128×128 이미지**로 군집화해야 합니다.

1. `image = data_1 @ data_2` 로 3840장을 재구성(128×128).
2. 표준화 후 **PCA(64차원)** 로 노이즈를 줄이고,
3. **KMeans(32)** 로 군집화.

캐글 점수: 베이스라인(원시 인자) ≈0.028 → **모범답안(재구성+PCA) ≈0.064** (리더보드 1위 ≈0.116).

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
a = np.load('data/data_1.npz')['arr_0']; b = np.load('data/data_2.npz')['arr_0']
N, K = len(a), 32

# (1) 재구성 이미지 (유일한 표현)
img = np.einsum('nij,njk->nik', a, b).reshape(N, -1)   # (3840, 128*128)
# (2) 표준화 + PCA
Xp = PCA(n_components=64, random_state=0).fit_transform(StandardScaler().fit_transform(img))
# (3) KMeans
labels = KMeans(K, n_init=10, random_state=0).fit_predict(Xp)
pd.DataFrame({'id': range(N), 'target': labels}).to_csv('submission.csv', index=False)
print('wrote submission.csv | cluster sizes(head):', np.bincount(labels)[:6])


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)